## Embeddings

Converting the text to vector representation

https://docs.langchain.com/oss/python/integrations/text_embedding

There are several embedding techniques

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" # Enable LangSmith tracing
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' # Set the LangSmith project name

In [2]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    api_key = API_KEY,
    base_url= BASE_URL,
    model = 'text-embedding-3-small', 
    # model = 'text-embedding-3-large', 
    dimensions = 1024
)
embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001F2503A8AD0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001F2503A9400>, model='text-embedding-3-small', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://lkm-ai-az-openai.openai.azure.com/openai/v1/', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [3]:
result = embedding_model.embed_query('This is an example of OpenAI Embedding')
print(result[:100])
print(len(result))

[0.0052947998046875, -0.006076812744140625, 0.0196685791015625, -0.013397216796875, -0.004367828369140625, -0.051788330078125, 0.02117919921875, 0.0196380615234375, -0.0164031982421875, 0.026611328125, 0.037994384765625, -0.0487060546875, -0.0160675048828125, -0.047637939453125, -0.01386260986328125, 0.002902984619140625, 0.0016260147094726562, -0.0185089111328125, -0.041015625, 0.007709503173828125, -0.0081787109375, -0.01654052734375, -0.00336456298828125, 0.052001953125, -0.004932403564453125, -0.019500732421875, 0.002498626708984375, 0.041839599609375, 0.0156097412109375, -0.01354217529296875, 0.0233612060546875, -0.0404052734375, 0.0019474029541015625, -0.018646240234375, -0.022430419921875, 0.0066680908203125, 0.017578125, 0.035736083984375, -0.04779052734375, 0.0472412109375, 0.045806884765625, -0.038787841796875, -0.037628173828125, 0.06903076171875, -0.023956298828125, 0.032684326171875, -0.0650634765625, -0.04827880859375, 0.008026123046875, 0.0626220703125, -0.060302734375, 

In [4]:
# Embedding docs - multiple at a time.
documents = [
    "This is a document about embeddings.",
    "Another document for embedding.",
    "Agentic AI is fun"
]

docs_embed = embedding_model.embed_documents(documents)
print(len(docs_embed))
print(len(docs_embed[0]))

for i, emd in enumerate(docs_embed):
    print(f'Doc {i} - embedding length {len(emd)}')

3
1024
Doc 0 - embedding length 1024
Doc 1 - embedding length 1024
Doc 2 - embedding length 1024


## Data Pipeline

In [12]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnableLambda

loader = RunnableLambda(lambda file_path: TextLoader(file_path).load(), name = 'TextLoader')
splitter = RunnableLambda(lambda docs: RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap= 20).split_documents(docs), name = 'TextSplitter')

embedding_model = OpenAIEmbeddings(
    api_key = API_KEY,
    base_url= BASE_URL,
    model = 'text-embedding-3-small', 
    # model = 'text-embedding-3-large', 
    dimensions = 1024
)

embedding_runnable = RunnableLambda(lambda chunks: embedding_model.embed_documents([doc.page_content for doc in chunks]), name = 'Embedding')

In [13]:
embedding_chain = (loader | splitter | embedding_runnable).with_config({'run_name': 'EmbeddingChain'})

In [14]:
embd_docs = embedding_chain.invoke('../data/FinAI.txt')

print(len(embd_docs))
print(len(embd_docs[0]))
print(embd_docs[0])

23
1024
[0.062164306640625, 0.027069091796875, 0.0545654296875, 0.066650390625, 0.01898193359375, -0.06329345703125, 0.00913238525390625, 0.04779052734375, -0.0654296875, 0.038909912109375, 0.0291900634765625, 0.0109405517578125, -0.0240631103515625, -0.053436279296875, 0.0282135009765625, -0.04388427734375, -0.0185699462890625, 0.01470184326171875, 0.057464599609375, 0.027587890625, 0.0282745361328125, 0.0260162353515625, -0.037139892578125, -0.01290130615234375, -0.0153350830078125, 0.037811279296875, 0.0196533203125, -0.00335693359375, 0.0170135498046875, -0.0283966064453125, 0.0701904296875, -0.020172119140625, -0.04376220703125, 0.0537109375, 0.024169921875, 0.033477783203125, 0.010650634765625, -0.032989501953125, 0.007297515869140625, 0.0297698974609375, -0.041534423828125, -0.058441162109375, -0.0263214111328125, 0.036834716796875, 0.057098388671875, -0.07611083984375, 0.004291534423828125, 0.0225067138671875, 0.016357421875, -0.00844573974609375, -0.079833984375, -0.0248718261

Now these docs can be pushed to the VDB.